This notebook cleans the charted data in `data/raw/included_sources_raw.csv` and checks it against the [codebook](../codebook.qmd). It writes `data/processed/included_sources_clean.csv` and `data/processed/bibliography.csv`, which holds the reference details of each included source, taken from the deduplicated records.

Cleaning trims whitespace, lowercases the closed variables, and sorts and deduplicates the values of the multi-valued variables. A value outside the allowed lists is reported and left as entered.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "included_sources_raw.csv"
DEDUP = ROOT / "data" / "processed" / "deduplicated_records.csv"
OUT = ROOT / "data" / "processed"

COLUMNS = [
    "source_id", "record_id", "tier", "field", "field_detail", "source_type",
    "peer_reviewed", "data_origin", "factor_types", "factor_categories",
    "level", "factors", "region", "notes",
]
REQUIRED = ["source_id", "record_id", "tier", "field", "source_type"]

# These lists mirror codebook.qmd. Change both together.
SINGLE = {
    "tier": ["1", "2"],
    "field": ["health", "other", "cross-disciplinary"],
    "source_type": ["empirical", "review", "policy-guidance", "opinion", "other"],
    "peer_reviewed": ["yes", "no"],
    "data_origin": ["research", "routine-reused", "mixed", "unclear"],
}
MULTI = {
    "factor_types": ["barrier", "facilitator", "governance-mechanism"],
    "factor_categories": ["technical", "motivational", "economic", "political", "legal", "ethical", "other"],
    "level": ["individual", "institutional", "system"],
}

raw = pd.read_csv(RAW, dtype=str).fillna("") if RAW.exists() else pd.DataFrame(columns=COLUMNS)
absent = [c for c in COLUMNS if c not in raw.columns]
if absent:
    raise ValueError("included_sources_raw.csv is missing column(s): " + ", ".join(absent))
have_data = len(raw) > 0
print(f"{len(raw)} charted source(s)." if have_data else "No charted sources yet. Add rows to data/raw/included_sources_raw.csv.")

No charted sources yet. Add rows to data/raw/included_sources_raw.csv.


In [2]:
if have_data:
    clean = raw[COLUMNS].apply(lambda s: s.str.strip())
    for col in SINGLE:
        clean[col] = clean[col].str.lower()

    def tidy(cell):
        values = [v.strip().lower() for v in cell.split(";") if v.strip()]
        return "; ".join(sorted(set(values)))

    for col in MULTI:
        clean[col] = clean[col].map(tidy)

## Checks

In [3]:
if have_data:
    issues = []
    for col, allowed in SINGLE.items():
        bad = clean.loc[(clean[col] != "") & ~clean[col].isin(allowed), col]
        issues += [(col, v, int(n)) for v, n in bad.value_counts().items()]
    for col, allowed in MULTI.items():
        vals = pd.Series([v for cell in clean[col] for v in cell.split("; ") if v], dtype=str)
        bad = vals[~vals.isin(allowed)]
        issues += [(col, v, int(n)) for v, n in bad.value_counts().items()]
    if issues:
        print("Values outside the codebook:")
        display(pd.DataFrame(issues, columns=["variable", "value", "n"]))
    else:
        print("All closed values match the codebook.")

    empty = {c: int((clean[c] == "").sum()) for c in REQUIRED if (clean[c] == "").any()}
    if empty:
        print("Required variables with empty cells:", empty)
    if clean["source_id"].duplicated().any():
        print("Some source_id values are repeated.")
    if DEDUP.exists():
        dedup = pd.read_csv(DEDUP, dtype=str).fillna("")
        unknown = set(clean["record_id"]) - set(dedup["record_id"])
        if unknown:
            print(f"{len(unknown)} record_id value(s) are not in deduplicated_records.csv.")

In [4]:
if have_data:
    clean.to_csv(OUT / "included_sources_clean.csv", index=False)
    print("Wrote data/processed/included_sources_clean.csv")
    if DEDUP.exists():
        keep = ["record_id", "authors", "year", "title", "journal", "doi"]
        bib = clean[["source_id", "record_id"]].merge(dedup[keep], on="record_id", how="left")
        bib.to_csv(OUT / "bibliography.csv", index=False)
        print("Wrote data/processed/bibliography.csv")
        display(bib.head(20))
    else:
        print("deduplicated_records.csv not found, so bibliography.csv was not written.")